In [1]:
from dataclasses import dataclass
from collections import defaultdict
from typing import List, Optional
import random

@dataclass
class HangmanResponse:
    guessed_character: str

class HangmanGame:    
    def __init__(self, target_word: str, max_lives: int = 6):
        self.target_word = target_word.lower()
        self.max_lives = max_lives
        self.lives_left = max_lives
        self.guessed_letters = set()
        self.incorrect_letters = set()
        self.correct_positions: List[Optional[str]] = [
            None if ch.isalpha() else ch for ch in self.target_word
        ]
        
        for i, ch in enumerate(self.target_word):
            if not ch.isalpha():
                self.correct_positions[i] = ch
    
    def guess(self, letter: str) -> tuple[bool, bool, bool]:
        letter = letter.lower()        
        if not letter.isalpha() or len(letter) != 1:
            return False, self.is_game_over(), self.is_winner()
       
        if letter in self.guessed_letters:
            return letter in self.correct_letters(), self.is_game_over(), self.is_winner()
        
        self.guessed_letters.add(letter)
        
        if letter in self.target_word:
            for i, ch in enumerate(self.target_word):
                if ch == letter:
                    self.correct_positions[i] = letter
            return True, self.is_game_over(), self.is_winner()
        else:
            self.lives_left -= 1
            self.incorrect_letters.add(letter)
            return False, self.is_game_over(), self.is_winner()
    
    def is_winner(self) -> bool:
        return all(pos is not None for pos in self.correct_positions)
    
    def is_game_over(self) -> bool:
        return self.is_winner() or self.lives_left <= 0
    
    def get_display_word(self) -> str:
        return " ".join("_" if pos is None else pos for pos in self.correct_positions)
    
    def get_incorrect_letters(self) -> List[str]:
        return sorted(self.incorrect_letters)
    
    def get_status(self) -> dict:
        return {
            "display_word": self.get_display_word(),
            "lives_left": self.lives_left,
            "incorrect_letters": self.get_incorrect_letters(),
            "guessed_letters": sorted(self.guessed_letters),
            "is_game_over": self.is_game_over(),
            "is_winner": self.is_winner()
        }

def solve_hangman_word(llm, word: str) -> None:
    game = HangmanGame(word, max_lives=6)    
    turn_number = 0

    while not game.is_game_over():
        prompt = f"""
        Current Guess: {game.get_display_word()}
        Lives Left: {game.lives_left}
        Incorrect Guesses: {','.join(game.get_incorrect_letters())}

        Provide your next single letter guess.
        """
        response = llm.prompt(prompt, schema=HangmanResponse)
        is_correct, game_over, winner = game.guess(response.guessed_character)        
        turn_number += 1
        
    return game.get_status()

In [2]:
word_buckets = defaultdict(list)
with open('/kaggle/input/datasets/ishita21gupta/english-words-corpus/words_alpha.txt', 'r') as f:
    for line in f:
        word = line.strip()
        word_buckets[len(word)].append(word)

def pick_word(length):
    return random.choice(word_buckets.get(length, []))